# ADVANCED · Restore the D1/D2 code pin (maintenance, run once)

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

Dataset 3 now has its own code pin (`<outputs>/D3_CODE_COMMIT.txt`, methodology log entry 45). Before that, D3
notebooks used the shared pin `CODE_COMMIT.txt`, which a D3 session could move to a Dataset 3 commit. This
notebook sets `CODE_COMMIT.txt` back to the last D1/D2 pin that predates Dataset 3, read from the pin's own
history (expected: `1b6988a`, the fixed-hyperparameter commit). The old lines stay in the file as history.

It changes **only that text file**: no graph, checkpoint or result is modified. It refuses (and changes nothing)
if any D1/D2 unit was completed under the current pin. Running it again is harmless (it then reports
"unchanged"). It runs the newest code without reading or writing any pin.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
# maintenance: runs the newest code of the branch and reads or writes no code pin
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--no-pin'])
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 1. Pins and the commits recorded by completed units (before)

In [ ]:
sh("python scripts/code_pin.py status")

## 2. Restore the D1/D2 pin

In [ ]:
sh("python scripts/code_pin.py restore-d1d2")